> ## SOLUTION / ANSWER KEY &mdash; Lab 8.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-sequential-pipeline.ipynb`](../lab-04-sequential-pipeline.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.4 &mdash; Sequential Pipeline of Specialists

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run the CS specialists in a fixed order over one customer ticket
- Feed each stage the previous stage's accumulated case
- See why a clean, ordered hand-off keeps each stage reliable

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Sequential — a pipeline of specialists](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-04"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The simplest collaboration is the **sequential pipeline** (deck slide 9): agents run in a **fixed order**,
each transforming the running case &mdash; for a support ticket that is **triage &rarr; billing &rarr;
tech**. It's a relay where the baton is the growing case. Each stage gets a **clean, focused input** (the
prior stage's output), so each does its narrow job well &mdash; and you can swap any stage independently.
(Watch out: errors **propagate** downstream, and it's **serial**, so latency adds up.)

In [ ]:
# Each stage is a specialist that takes the running case and returns it, extended with its note.
print("pipeline: triage -> billing -> tech")

## Build it
Complete `run_pipeline` so each stage receives the previous stage's output.

In [ ]:
def run_pipeline(ticket, stages):
    case = ticket
    trail = []
    for stage in stages:
        case = stage(case)
        trail.append(case)
    return {"case": case, "trail": trail}

STAGES = [
    lambda c: c + " | triage -> billing+tech",
    lambda c: c + " | billing: duplicate charge on 4471",
    lambda c: c + " | tech: matches BUG-231",
]

try:
    out = run_pipeline("ticket: charged twice, app crashing", STAGES)
    print("final case:", out["case"])
    for step in out["trail"]:
        print("  step:", step)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The ticket passes through **every stage in order**; each `trail` entry **builds on** the previous one.
- Triage runs before the specialists, so the specialists get a pre-sorted case &mdash; a clean, focused input.
- Because stages are just functions of the running case, you can **swap** any one independently. Errors, though, propagate downstream.

## Your turn (open task &mdash; no grader)
Insert a new **`policy`** stage between billing and tech (e.g. `lambda c: c + " | policy: refund within 30 days"`)
and re-run. **What good looks like:** the new stage appears in order in the trail, the final case carries every
stage's note, and removing a stage cleanly shortens the pipeline &mdash; no other stage needs to change.

---
### Key takeaway
A pipeline is the multi-agent version of Module 7's automation pipeline -- each stage a specialist over the same ticket. Clean, ordered hand-offs make each stage reliable; just remember errors propagate downstream.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>